## 1 — Môi trường

In [ ]:
import sys, os, platform, subprocess
from pathlib import Path
print('Python:', platform.python_version())
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
print('RAM bytes:', os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES'))


## 2 — Secret

In [ ]:
from kaggle_secrets import UserSecretsClient
try:
    token = UserSecretsClient().get_secret('TABPFN_TOKEN')
except Exception:
    raise RuntimeError('Thiếu Kaggle Secret TABPFN_TOKEN; thêm và cấp quyền notebook.') from None
if not token or not token.strip():
    raise RuntimeError('Kaggle Secret TABPFN_TOKEN rỗng')
os.environ['TABPFN_TOKEN'] = token
del token
os.environ['TABPFN_NO_BROWSER'] = '1'
print('TABPFN_TOKEN configured:', bool(os.environ.get('TABPFN_TOKEN')))


## 3 — Cài dependencies

In [ ]:
# Có thể chỉ định đường dẫn nếu attach nhiều bundle.
DATA_ROOT = None
candidates = list(Path('/kaggle/input').rglob('metadata/bundle_manifest.json'))
if DATA_ROOT is None:
    if len(candidates) != 1:
        raise RuntimeError('Cần đúng một bundle hoặc đặt DATA_ROOT rõ ràng')
    DATA_ROOT = candidates[0].parents[1]
DATA_ROOT = Path(DATA_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(DATA_ROOT/'requirements.txt')], check=True)


## 4 — Định vị dữ liệu và output

In [ ]:
WORK = Path('/kaggle/working')
print('DATA_ROOT:', DATA_ROOT)
sys.path.insert(0, str(DATA_ROOT/'scripts'))
from notebook_steps import full_after_probe


## 5 — Kiểm tra bundle; lỗi sẽ dừng

In [ ]:
subprocess.run([sys.executable, str(DATA_ROOT/'scripts/verify_bundle.py'), '--data-root', str(DATA_ROOT)], check=True)


## 6 — Probe v3 chỉ train

In [ ]:
subprocess.run([sys.executable, str(DATA_ROOT/'scripts/resource_probe.py'), '--version', 'v3', '--data-root', str(DATA_ROOT), '--output-dir', str(WORK/'resource_probe_v3.json')], check=True)


## 7 — Full21K v3

In [ ]:
full_after_probe(DATA_ROOT, WORK, 'v3')


## 8 — Giải phóng bộ nhớ

In [ ]:
# Mô hình nằm trong subprocess đã kết thúc; không giữ V3 cùng V3.5.
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 9 — Probe v3_5 chỉ train

In [ ]:
subprocess.run([sys.executable, str(DATA_ROOT/'scripts/resource_probe.py'), '--version', 'v3_5', '--data-root', str(DATA_ROOT), '--output-dir', str(WORK/'resource_probe_v35.json')], check=True)


## 10 — Full21K v3_5

In [ ]:
full_after_probe(DATA_ROOT, WORK, 'v3_5')


## 11 — Đóng gói kết quả

In [ ]:
subprocess.run([sys.executable, str(DATA_ROOT/'scripts/package_results.py'), '--data-root', str(DATA_ROOT), '--output-dir', str(WORK)], check=True)
